# Module 4 Dataset Exploration: Amod Counseling Q&A

This notebook explores Hugging Face dataset `Amod/mental_health_counseling_conversations` before using it in retrieval. The goal is to understand the dataset shape, repeated questions, text lengths, and cleaning needs.

## 1. Setup

Keep imports simple and readable. The notebook saves a cleaned JSON file locally and a small summary report.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
pd.set_option("display.max_colwidth",None)
from datasets import load_dataset

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports" / "module_4_rag_retrieval"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Dataset

The dataset has two columns: `Context` for the user question/problem and `Response` for a counselor-style answer. The cleaned output renames them to `question` and `answer` for readability.

In [ ]:
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")
df = dataset.to_pandas()

print(dataset)
df.head()

## 3. Rename Columns

The original dataset columns are `Context` and `Response`. For easier explanation, we rename them to `question` and `answer`.

In [ ]:
df = df.rename(columns={"Context": "question", "Response": "answer"})
print(df.shape)
df.head(1)

## 4. Basic Cleaning

This keeps the original meaning but removes repeated whitespace and strange spacing.

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"([.!?])(?=[A-Z])", r"\1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["question"] = df["question"].apply(clean_text)
df["answer"] = df["answer"].apply(clean_text)

df[["question", "answer"]].head(2)

## 5. Missing, Empty, And Duplicate Checks

This tells us whether the dataset can be used directly or needs filtering.

In [ ]:
quality_checks = {
    "raw_rows": int(len(df)),
    "empty_questions": int((df["question"] == "").sum()),
    "empty_answers": int((df["answer"] == "").sum()),
    "very_short_answers": int((df["answer_words"] < MIN_ANSWER_WORDS).sum()),
    "exact_duplicate_rows": int(df.duplicated(subset=["question", "answer"]).sum()),
    "unique_questions_before_cleaning": int(df["question"].nunique()),
    "unique_answers_before_cleaning": int(df["answer"].nunique()),
}

quality_checks

## 6. Text Length Analysis

Length matters because long answers may need chunking, while very short answers may not be useful for retrieval.

In [ ]:
df["question_words"] = df["question"].str.split().str.len()
df["answer_words"] = df["answer"].str.split().str.len()

length_summary = df[["question_words", "answer_words"]].describe().round(2)
length_summary

## 7. Repeated Questions

This is one of the dataset secrets: many user questions appear more than once with different counselor answers.

In [ ]:
question_counts = df["question"].value_counts()
repeated_questions = question_counts[question_counts > 1]

print("Repeated questions:", len(repeated_questions))
print("Rows belonging to repeated questions:", int(df["question"].isin(repeated_questions.index).sum()))

repeated_questions.head(10)

## 8. Example Of A Repeated Question

This shows why the dataset is not a typical document corpus. One user problem can have many possible answers.

In [ ]:
example_question = repeated_questions.index[0]
example_rows = df[df["question"] == example_question][["question", "answer"]]

print("Number of answers for this question:", len(example_rows))
print("Question:\n", example_question[:1000])

example_rows["answer"].head(5).to_list()

## 9. Short And Long Answers

Short answers may be weak retrieval documents. Very long answers may need chunking later.

In [ ]:
short_answers = df.sort_values("answer_words").head(5)[["question", "answer", "answer_words"]]
long_answers = df.sort_values("answer_words", ascending=False).head(5)[["question", "answer", "answer_words"]]

short_answers

In [ ]:
long_answers

## 10. Build A Clean Q&A Dataset

For now, keep one row per Q&A pair. Add IDs and simple metadata. Later we can decide whether to group repeated contexts.

In [ ]:
MIN_ANSWER_WORDS = 25

clean_df = df.copy()
clean_df = clean_df[clean_df["question"] != ""]
clean_df = clean_df[clean_df["answer"] != ""]
clean_df = clean_df[clean_df["answer_words"] >= MIN_ANSWER_WORDS]
clean_df = clean_df.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)

question_group_sizes = clean_df["question"].value_counts()
clean_df["question_group_size"] = clean_df["question"].map(question_group_sizes)
clean_df["qa_id"] = [f"amod_qa_{i:04d}" for i in range(1, len(clean_df) + 1)]
clean_df["source"] = "Amod/mental_health_counseling_conversations"

clean_df = clean_df[[
    "qa_id",
    "source",
    "question",
    "answer",
    "question_words",
    "answer_words",
    "question_group_size",
]]

clean_df.head()

## 11. Save Clean Dataset And Summary

The processed dataset is saved locally under `data/processed`. The summary report is saved under `reports/module_4_rag_retrieval`.

In [ ]:
clean_path = PROCESSED_DIR / "amod_clean_qa.json"
summary_path = REPORT_DIR / "amod_dataset_summary.json"

clean_records = clean_df.to_dict(orient="records")
clean_path.write_text(json.dumps(clean_records, indent=2, ensure_ascii=False), encoding="utf-8")

summary = {
    "dataset": "Amod/mental_health_counseling_conversations",
    "raw_rows": int(len(df)),
    "clean_rows": int(len(clean_df)),
    "quality_checks": quality_checks,
    "unique_questions": int(clean_df["question"].nunique()),
    "unique_answers": int(clean_df["answer"].nunique()),
    "repeated_question_count": int((question_group_sizes > 1).sum()),
    "rows_with_repeated_question": int((clean_df["question_group_size"] > 1).sum()),
    "min_question_words": int(clean_df["question_words"].min()),
    "max_question_words": int(clean_df["question_words"].max()),
    "average_question_words": round(float(clean_df["question_words"].mean()), 2),
    "min_answer_words": int(clean_df["answer_words"].min()),
    "max_answer_words": int(clean_df["answer_words"].max()),
    "average_answer_words": round(float(clean_df["answer_words"].mean()), 2),
    "minimum_answer_words_filter": MIN_ANSWER_WORDS,
    "top_repeated_question_group_sizes": [int(value) for value in question_group_sizes.head(10).to_list()],
    "main_observation": "The dataset is Q&A/counseling-case shaped, with many repeated questions and multiple possible answers per user concern.",
}

summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Saved clean dataset to: {clean_path}")
print(f"Saved summary report to: {summary_path}")
summary

## 12. Practical Interpretation

This dataset should not be treated like a normal document corpus. It is better understood as a collection of counseling cases. For retrieval, there are two reasonable options:

1. Keep one Q&A pair per record and retrieve similar cases.
2. Group repeated questions and summarize answer patterns later.

For the first Module 4 version, keep the cleaned Q&A records simple. Later, combine them with the CCI information-sheet corpus for richer retrieval.